In [1]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import evaluate

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

In [3]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

dataset

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [4]:
print(dataset)

print("\nFirst review:\n")
print(dataset["train"][0]["text"])

print("\nLabel:")
print(dataset["train"][0]["label"])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

First review:

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicia

In [5]:
small_train = dataset["train"].shuffle(seed=42).select(range(3000))
small_test = dataset["test"].shuffle(seed=42).select(range(1000))

In [6]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test = small_test.map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
import evaluate
import numpy as np
accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [9]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./distilbert-imdb-results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True
)

In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.330460,0.315247,0.866000
2,0.231306,0.428580,0.866000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=0.31381212361653643, metrics={'train_runtime': 160.7124, 'train_samples_per_second': 37.334, 'train_steps_per_second': 4.667, 'total_flos': 397402195968000.0, 'train_loss': 0.31381212361653643, 'epoch': 2.0})

In [12]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy
0.231306,0.315247,2,0.866000


{'eval_loss': 0.31524723768234253, 'eval_accuracy': 0.866}

In [13]:
sentiment_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

sentiment_pipeline("This movie was amazing and I really enjoyed it.")

[{'label': 'LABEL_1', 'score': 0.9682776927947998}]

In [14]:
sentiment_pipeline("The movie was boring and the story was weak.")

[{'label': 'LABEL_0', 'score': 0.9761863946914673}]

In [15]:
model.save_pretrained("./distilbert-imdb-sentiment-model")
tokenizer.save_pretrained("./distilbert-imdb-sentiment-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./distilbert-imdb-sentiment-model/tokenizer_config.json',
 './distilbert-imdb-sentiment-model/tokenizer.json')

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
import os

project_path = "/content/drive/MyDrive/AI_Projects/DistilBERT_IMDB_Sentiment"

os.makedirs(project_path, exist_ok=True)

print("Project folder:", project_path)

Project folder: /content/drive/MyDrive/AI_Projects/DistilBERT_IMDB_Sentiment


In [18]:
model_path = f"{project_path}/model"

model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print("Model saved to:", model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/AI_Projects/DistilBERT_IMDB_Sentiment/model
